<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5509-notebooks/main/_banners/opim5509_banner.svg" width="100%" alt="OPIM 5509 banner"/>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drdave-teaching/OPIM5509-notebooks/blob/main/Module4/Forecasting_Electricity_Demand_RNN.ipynb)

# Forecasting Electricity Demand with an RNN
--------------------------------------------------
**Dr. Dave Wanik - University of Connecticut**

You work for the utility. Every hour, the grid has to serve whatever Connecticut demands - and the dispatch desk wants to know **tomorrow's hourly load** today. This is the Assignment 2 dataset (hourly demand plus Bradley airport weather, 2011-2021), now treated the way it deserves: as a **sequence**.

By the end you'll have: the three dumb baselines every forecast must beat, a univariate LSTM (last 24 hours -> next hour), a multivariate LSTM that also sees the weather, a **24-hour-ahead** multi-step model, and a peak-hour classifier - all on the same business question.

🔴
<!-- 🎙 DAVE TALKING POINTS — M4 · 16 — Forecasting electricity demand, Pt 1: data, baselines, a univariate LSTM
- The business question: you are the utility, forecast tomorrow's hourly load. Same dataset as Assignment 2 - now as a SEQUENCE.
- The file is UNSORTED - show is_monotonic_increasing == False before and after the sort. Never trust that a time series arrives in order.
- EDA beats: one July week (daily peaks at 6 PM), hour-of-day profile (2,790 at midnight -> ~3,990 at 6 PM), month profile (summer AC and winter heat, cheap shoulders), demand vs temperature is a U (heating on the left, cooling on the right), and the 2020 COVID dip = distribution shift.
- Chronological split: 2017-2018 train, 2019 test. No shuffle - say why.
- Baselines FIRST: mean-only (554 MW), same-hour-yesterday (227), last-hour persistence (129). Persistence at one hour ahead is brutal - that's the villain.
- Univariate LSTM: scale on train only, look-back 24 -> next hour, LSTM(32) + Dense(1), early stopping. Read the MAE against 129: the LSTM lands at ~44 MW - it beats persistence by 3x because two years of history taught it the daily shape, not just the last value.
-->


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, classification_report, confusion_matrix
from keras.models import Sequential, load_model
from keras.layers import LSTM, Dense, Dropout
from keras.callbacks import EarlyStopping
import keras
keras.utils.set_random_seed(5509)   # reproducibility: same numbers every run (CPU exact; a GPU may drift a little)

## Read in the data - and sort it!

The file is not in date order. `df.info()` won't tell you that; you have to ask.

In [ ]:
url = "https://raw.githubusercontent.com/drdave-teaching/OPIM5509Files/main/OPIM5509_Module4_Files/data/BDL_cleanweather_energy.csv"
df = pd.read_csv(url, parse_dates=["Datetime"])
print("rows:", len(df), "| in date order?", df["Datetime"].is_monotonic_increasing)

df = df.sort_values("Datetime").set_index("Datetime")
print("after sorting - in date order?", df.index.is_monotonic_increasing, "| span:", df.index.min().date(), "->", df.index.max().date())

df = df.ffill()          # 0.3% of the weather rows are missing - carry the last reading forward (fine at hourly cadence)
df.info()

In [ ]:
df.describe().round(1).T

## EDA: the shape of demand

Before any model - what does a week look like, what does a day look like, what does a year look like, and what does temperature do to it?

In [ ]:
# one summer week - daily peaks in the evening, lower weekend load
wk = df.loc["2019-07-08":"2019-07-14", "Demand"]
wk.plot(figsize=(11, 3), title="One week of hourly demand (MW), July 2019"); plt.ylabel("MW"); plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
df.groupby(df.index.hour)["Demand"].mean().plot(ax=ax[0], marker="o", title="Mean demand by hour of day"); ax[0].set_xlabel("hour"); ax[0].set_ylabel("MW")
df.groupby(df.index.month)["Demand"].mean().plot(ax=ax[1], marker="o", title="Mean demand by month"); ax[1].set_xlabel("month")
plt.tight_layout(); plt.show()

In [ ]:
# demand vs temperature is a U: heating on the cold side, air conditioning on the hot side
s = df.sample(6000, random_state=5509)
plt.figure(figsize=(6, 4)); plt.scatter(s["BDL_tmpf"], s["Demand"], s=4, alpha=0.4)
plt.xlabel("temperature (F)"); plt.ylabel("demand (MW)"); plt.title("Demand vs temperature"); plt.show()

In [ ]:
# the 2020 dip: the world changed and the model's data didn't - that's "distribution shift"
df.groupby(df.index.year)["Demand"].mean().plot(kind="bar", figsize=(8, 3), title="Mean demand by year (MW)"); plt.show()

## Slice and split - chronologically

Three years is plenty for the lecture and fast to train on. **Train on 2017-2018, test on 2019** - the future is never in the training set.

In [ ]:
data  = df.loc["2017-01-01":"2019-12-31"].copy()
train = data.loc[:"2018-12-31"]
test  = data.loc["2019-01-01":]
print("train:", train.shape, "| test:", test.shape)

## Baselines first

A forecast is only good relative to the dumb thing you could have done instead. Three dumb things, scored on the **test** year:

In [ ]:
y = test["Demand"]
baselines = pd.Series({
    "mean-only (predict the train average)": mean_absolute_error(y, np.full(len(y), train["Demand"].mean())),
    "seasonal naive (same hour yesterday)":  mean_absolute_error(y.iloc[24:], y.shift(24).iloc[24:]),
    "persistence (same as last hour)":       mean_absolute_error(y.iloc[1:],  y.shift(1).iloc[1:]),
}, name="test MAE (MW)").round(1)
baselines

## Univariate LSTM: the last 24 hours -> the next hour

Same `split_sequence` as the temperature notebook. Scale on **train only** (the scaler is part of the model), window the scaled series, reshape to `(samples, look-back, 1)`.

<!-- WINDOW-FN -->
## The window-making function: `split_sequence`

Chops **one column** into overlapping windows.

- **`n_steps`** is the **look-back**: each input X is `n_steps` values in a row.
- The target y is the **next** value after the window.

With `[10, 20, 30, 40, 50]` and `n_steps=3`: `[10, 20, 30] → 40`, then `[20, 30, 40] → 50`. The result is `(samples, n_steps)`, and we add a trailing `1` for Keras: `(samples, n_steps, 1)`.


In [ ]:
def split_sequence(seq, n_steps):
    X, y = [], []
    for i in range(len(seq) - n_steps):
        X.append(seq[i:i+n_steps]); y.append(seq[i+n_steps])
    return np.array(X), np.array(y)

n_steps = 24                                   # one day of history
sc_y = MinMaxScaler().fit(train[["Demand"]])   # fit on TRAIN only
tr_s = sc_y.transform(train[["Demand"]]).ravel()
te_s = sc_y.transform(test[["Demand"]]).ravel()

X_tr, y_tr = split_sequence(tr_s, n_steps)
X_te, y_te = split_sequence(te_s, n_steps)
X_tr = X_tr.reshape(X_tr.shape[0], n_steps, 1)   # (samples, look-back, features=1) - the 3-D tensor
X_te = X_te.reshape(X_te.shape[0], n_steps, 1)
print("train tensor:", X_tr.shape, "| test tensor:", X_te.shape)

In [ ]:
es = EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True, verbose=1)

uni = Sequential([LSTM(32, input_shape=(n_steps, 1)), Dense(1)])
uni.compile(optimizer="adam", loss="mse", metrics=["mae"])
uni.summary()
hist = uni.fit(X_tr, y_tr, epochs=50, batch_size=64, validation_split=0.2, callbacks=[es], verbose=1)

In [ ]:
plt.plot(hist.history["loss"], label="train"); plt.plot(hist.history["val_loss"], label="validation")
plt.title("Univariate LSTM - loss"); plt.legend(); plt.show()

pred_uni = sc_y.inverse_transform(uni.predict(X_te, verbose=0)).ravel()   # back to MW
y_true   = sc_y.inverse_transform(y_te.reshape(-1, 1)).ravel()
mae_uni  = mean_absolute_error(y_true, pred_uni)
print(f"univariate LSTM test MAE: {mae_uni:.1f} MW   (persistence: {baselines.iloc[2]:.1f})")

In [ ]:
# look at a test week: the model vs reality
i0 = 24*7*26   # a week in mid-year
plt.figure(figsize=(11, 3))
plt.plot(y_true[i0:i0+24*7], label="actual"); plt.plot(pred_uni[i0:i0+24*7], label="LSTM (1h ahead)")
plt.title("One test week - univariate LSTM"); plt.ylabel("MW"); plt.legend(); plt.show()

🔴
<!-- 🎙 DAVE TALKING POINTS — M4 · 17 — Forecasting electricity demand, Pt 2: add the weather, forecast 24 hours ahead, call the peak
- Multivariate: temperature, dew point, humidity, plus hour-of-day and day-of-week as sin/cos (the clock is a circle: hour 23 is next to hour 0). Demand goes LAST - split_sequences takes the target from the right, and demand's own history STAYS in X (the window method taught you that last hour is the best feature).
- Does the weather + clock help at ONE hour ahead? Here yes: ~44 -> ~34 MW. Say why it isn't more - the last hour already carries most of the weather's effect.
- 24 hours ahead is where it matters: past week -> next 24 hours, Dense(24). Plot MAE by horizon (~167 MW averaged over the day vs 227 seasonal-naive); the fair baseline a day ahead is seasonal-naive, not persistence.
- Peak-hour classification: top-10% hours, sigmoid head, majority baseline is 91.5% - so accuracy is useless, read precision/recall for the peaks (~0.91 / ~0.88).
- Close on reproducibility: the seed, and save -> load_model -> identical predictions. This is the notebook students copy for their projects.
-->


## Add the weather (multivariate)

The model gets to see temperature, dew point, humidity - and the **clock**, encoded as sine/cosine so hour 23 sits next to hour 0. Demand is the **last column** because `split_sequences` takes the target from the right - and demand's own past stays in the inputs, because *what demand did last hour* is the single most useful feature.

<!-- WINDOW-FN -->
## The window-making function: `split_sequences`

Target in the **last column**, then:

- **`n_steps`** is the **look-back**: X is the last `n_steps` rows of **every column, the target included**. Yesterday's demand is the most useful input there is.
- y is the target **one step** after the window ends.

Result: X is `(samples, n_steps, n_features)`.


In [ ]:
def add_clock(d):
    d = d.copy()
    d["hour_sin"] = np.sin(2*np.pi*d.index.hour/24);      d["hour_cos"] = np.cos(2*np.pi*d.index.hour/24)
    d["dow_sin"]  = np.sin(2*np.pi*d.index.dayofweek/7);  d["dow_cos"]  = np.cos(2*np.pi*d.index.dayofweek/7)
    return d

feats = ["BDL_tmpf", "BDL_dwpf", "BDL_relh", "hour_sin", "hour_cos", "dow_sin", "dow_cos", "Demand"]   # Demand LAST
trm, tem = add_clock(train)[feats], add_clock(test)[feats]

sc_X = MinMaxScaler().fit(trm)                 # fit on TRAIN only
trm_s, tem_s = sc_X.transform(trm), sc_X.transform(tem)

def split_sequences(seqs, n_steps):
    X, y = [], []
    for i in range(len(seqs) - n_steps):
        X.append(seqs[i:i+n_steps, :]); y.append(seqs[i+n_steps, -1])   # inputs = EVERY column (demand's own past included); target = last col, NEXT step
    return np.array(X), np.array(y)

Xm_tr, ym_tr = split_sequences(trm_s, n_steps)
Xm_te, ym_te = split_sequences(tem_s, n_steps)
n_features = Xm_tr.shape[2]
print("multivariate train tensor:", Xm_tr.shape, "-> (samples, look-back, features)")

In [ ]:
multi = Sequential([LSTM(32, input_shape=(n_steps, n_features)), Dense(1)])
multi.compile(optimizer="adam", loss="mse", metrics=["mae"])
hist_m = multi.fit(Xm_tr, ym_tr, epochs=50, batch_size=64, validation_split=0.2, callbacks=[es], verbose=1)

pred_multi = sc_y.inverse_transform(multi.predict(Xm_te, verbose=0)).ravel()   # Demand scaler = the last column's scaler
mae_multi  = mean_absolute_error(y_true, pred_multi)
pd.Series({"persistence": baselines.iloc[2], "univariate LSTM": mae_uni, "multivariate LSTM (+weather, +clock)": mae_multi},
          name="test MAE, 1 hour ahead (MW)").round(1)

## 24 hours ahead (multi-step)

One hour ahead is nearly free - the last hour already tells you almost everything. The dispatch desk needs **tomorrow**: from the past week, predict the next 24 hours at once (`Dense(24)`). Now the honest baseline is *same hour yesterday*, and the interesting picture is **error by horizon**.

<!-- WINDOW-FN -->
## The window-making function: `split_multistep`

For forecasting **several steps at once**:

- **`lookback`**: how many past rows go in (X is `(samples, lookback, n_features)`).
- **`horizon`**: how many future values come out (y is `(samples, horizon)`), one output per step, so the model ends in `Dense(horizon)`.

With `lookback=168, horizon=24`: the past week in, the next day out.


In [ ]:
def split_multistep(seqs, lookback, horizon):
    X, y = [], []
    for i in range(len(seqs) - lookback - horizon + 1):
        X.append(seqs[i:i+lookback, :]); y.append(seqs[i+lookback:i+lookback+horizon, -1])
    return np.array(X), np.array(y)

lookback, horizon = 24*7, 24                          # past week -> next day
Xs_tr, ys_tr = split_multistep(trm_s, lookback, horizon)
Xs_te, ys_te = split_multistep(tem_s, lookback, horizon)
print("multi-step tensors:", Xs_tr.shape, "->", ys_tr.shape)

step = Sequential([LSTM(64, input_shape=(lookback, n_features)), Dense(horizon)])
step.compile(optimizer="adam", loss="mse", metrics=["mae"])
hist_s = step.fit(Xs_tr, ys_tr, epochs=50, batch_size=64, validation_split=0.2, callbacks=[es], verbose=1)

In [ ]:
pred_s = sc_y.inverse_transform(step.predict(Xs_te, verbose=0))          # (samples, 24) in MW
true_s = sc_y.inverse_transform(ys_te)
mae_by_h = np.abs(pred_s - true_s).mean(axis=0)                            # MAE at horizon 1..24

# fair baseline at every horizon: same hour yesterday
dem_te = tem["Demand"].values
naive_by_h = np.array([mean_absolute_error(dem_te[lookback+h:len(dem_te)-horizon+h+1], dem_te[lookback+h-24:len(dem_te)-horizon+h+1-24]) for h in range(horizon)])

plt.figure(figsize=(8, 3.5))
plt.plot(range(1, 25), mae_by_h, marker="o", label="LSTM, past week -> next 24h")
plt.plot(range(1, 25), naive_by_h, marker="s", label="seasonal naive (same hour yesterday)")
plt.xlabel("hours ahead"); plt.ylabel("test MAE (MW)"); plt.title("Error grows with the horizon - does the model still beat the naive forecast?"); plt.legend(); plt.show()
print(f"24h-ahead model MAE averaged over horizons: {mae_by_h.mean():.1f} MW   | seasonal naive: {naive_by_h.mean():.1f} MW")

In [ ]:
# one test day, all 24 hours at once
d = 200
plt.figure(figsize=(8, 3)); plt.plot(true_s[d], marker="o", label="actual"); plt.plot(pred_s[d], marker="o", label="forecast made 24h earlier")
plt.xlabel("hour of the forecast day"); plt.ylabel("MW"); plt.legend(); plt.title("A day-ahead forecast"); plt.show()

## Bonus: is this a peak hour? (classification)

The desk also wants a flag: *will this hour be one of the expensive ones?* Define a peak as the **top 10% of training-set demand**, put a **sigmoid** on the same multivariate windows, and read precision/recall - the majority-class baseline is 90%, so accuracy is useless here.

In [ ]:
thr = train["Demand"].quantile(0.90)
peak_tr = (sc_y.inverse_transform(ym_tr.reshape(-1, 1)).ravel() > thr).astype(int)
peak_te = (y_true > thr).astype(int)
print(f"peak threshold: {thr:.0f} MW | peak hours in test: {peak_te.mean():.1%}  (majority baseline = {1-peak_te.mean():.1%} by always saying 'no')")

clf = Sequential([LSTM(32, input_shape=(n_steps, n_features)), Dense(1, activation="sigmoid")])
clf.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
clf.fit(Xm_tr, peak_tr, epochs=50, batch_size=64, validation_split=0.2, callbacks=[es], verbose=0)

p = (clf.predict(Xm_te, verbose=0).ravel() > 0.5).astype(int)
print(classification_report(peak_te, p, target_names=["normal hour", "PEAK hour"]))
print(confusion_matrix(peak_te, p))

## Save the model and use it again

Reproducibility is a seed **and** a saved artifact. Save the day-ahead model, reload it, prove it gives the same forecast.

In [ ]:
step.save('Forecasting_Electricity_Demand_RNN_dayahead.keras')
reloaded = load_model('Forecasting_Electricity_Demand_RNN_dayahead.keras')
print("reloaded model reproduces the forecast:", np.allclose(step.predict(Xs_te[:5], verbose=0), reloaded.predict(Xs_te[:5], verbose=0)))

## On your own

- Change the look-back (6 hours? 3 days?) and watch what happens to the one-hour and 24-hour errors.
- Add a **holiday flag** - the model has never been told that July 4th isn't a Tuesday.
- Swap `LSTM` for `GRU`, then for a `Bidirectional(LSTM(...))`. One-word changes - same shapes.
- Forecast a **heat-wave week** (try mid-July 2019). Where does the model miss - the peak height, or the timing?
- Train on 2011-2019 and test on **2020**. That's what distribution shift does to a forecast.